1) Positiv/Negativ-Klassifikator 

Überwachte Klassifikation: Deployment-KPIs → Label `1` (negativ/mutiert) und `0` (positiv).

Dient als Proof of Concept, ob die Snippets anhand der kpi Metriken in positiv und negatibeispiele eingeteilt werden können. 


In [1]:
import numpy as np
import pandas as pd
import sklearn

import pathlib


from sklearn.model_selection import GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [2]:
ROOT = pathlib.Path("/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment")
KPI_CSV_DIR = ROOT / "export_kpis" / "csv_kpis"
KPI_CSV_CANDIDATES = sorted(KPI_CSV_DIR.glob("all_projects_kpis_[0-9]*.csv"))
if not KPI_CSV_CANDIDATES:
 raise FileNotFoundError(
 "Keine Zeitstempel-KPI-CSV in export_kpis/csv_kpis/ gefunden - zuerst KPIs aus dem Bucket ziehen."
)
KPI_CSV = KPI_CSV_CANDIDATES[-1]

df = pd.read_csv(KPI_CSV)

is_neg = df["variant"].str.endswith("_neg")
neg = is_neg.sum()
pos = (~is_neg).sum()
print("Zeile insgesammt", len(df))
print("Davon positive Datensaetze :", pos)
print("Negative Datensaetze :", neg)

if neg == 0 or pos == 0:
    raise ValueError(
        "Entweder keine positiven oder negativen datensaetze vorhanden"
    ) 

Zeile insgesammt 750
Davon positive Datensaetze : 600
Negative Datensaetze : 150


2) Label & Feature Matrix 

- label is_neg -> negativ Endung -> wird zu 1 (negativ)
- project_id : Projekt ohne _neg endung  -> 0 (positiv)
- pair-id : Pärchen Gruppen -> pos+neg Code bleiben zusammen
 

In [3]:
df["is_neg"] = df["variant"].str.endswith("_neg").astype(int)
df["project_id"] = df["variant"].str.replace("_neg","", regex=False)
df["pair_id"] = df["target_method"]

In [4]:
print("Spalten:", list(df.columns))
print()
print("Projekte:", sorted(df["project_id"].unique()))
print("Umgebungen:", sorted(df["env"].unique()))
print()
metric_cols = ["avg_latency_ms", "error_rate_percent", "p95_latency_ms", "requests_per_sec", "total_failures", "total_requests"]
print(df[metric_cols].describe().T)
print()
print("Fehlende Werte:")
print(df[metric_cols].isna().sum())
print()
print("Zeilen je Projekt und Label:")
print(df.groupby(["project_id", "is_neg"]).size().unstack(fill_value=0))

Spalten: ['variant', 'env', 'source', 'avg_latency_ms', 'error_rate_percent', 'p95_latency_ms', 'requests_per_sec', 'target_method', 'total_failures', 'total_requests', 'is_neg', 'project_id', 'pair_id']

Projekte: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13']
Umgebungen: ['extreme', 'high', 'low', 'medium', 'prod']

                    count         mean           std    min       25%  \
avg_latency_ms      750.0  3468.312187   6041.606515   3.22   50.4125   
error_rate_percent  750.0     2.887813     12.963596   0.00    0.0000   
p95_latency_ms      750.0  7586.838667  13781.170272   3.00  190.0000   
requests_per_sec    750.0    49.262413     52.174065   0.34    3.8500   
total_failures      750.0    26.880000    210.139267   0.00    0.0000   
total_requests      750.0  3884.965333   5365.242799  10.00  295.0000   

                         50%        75%       max  
avg_latency_ms       431.280  4957.3025  44660.62  
error_rate_percent 

In [5]:
RANDOM_STATE = 42
TARGET = "is_neg"

TRAINING_PROJECTS = ["P01", "P02", "P04", "P05", "P06", "P07", "P08", "P09", "P10", "P11"]
EVAL_PROJECTS = ["P12", "P13"]

NUM_FEATURES = [
    "avg_latency_ms",
    "error_rate_percent",
    "p95_latency_ms",
    "requests_per_sec",
]
LOG_FEATURES =["avg_latency_ms", "p95_latency_ms", "requests_per_sec"]
PLAIN_FEATURES = ["error_rate_percent"]
CAT_FEATURES = ["env"]

print("Trainings-Projekte (P01-P11 ohne P03):", TRAINING_PROJECTS)
print("Held-out Evaluations-Projekte:", EVAL_PROJECTS)
print("Numerische Features:", NUM_FEATURES)
print("Kategorische Features:", CAT_FEATURES)

Trainings-Projekte (P01-P11 ohne P03): ['P01', 'P02', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11']
Held-out Evaluations-Projekte: ['P12', 'P13']
Numerische Features: ['avg_latency_ms', 'error_rate_percent', 'p95_latency_ms', 'requests_per_sec']
Kategorische Features: ['env']


3) Training und Evaluation Datenset aufteilen 


In [6]:
train_val = df[df["project_id"].isin(TRAINING_PROJECTS)]
test_df = df[df["project_id"].isin(EVAL_PROJECTS)]

shufflesplit = GroupShuffleSplit (n_splits=1,test_size=0.25,random_state=RANDOM_STATE)
train_idx,val_idx = next(shufflesplit.split(train_val, groups= train_val["pair_id"]))

train_df = train_val.iloc[train_idx].copy()
val_df = train_val.iloc[val_idx].copy()

print("KPI-CSV", KPI_CSV.name)
print("Train-Projekte:", sorted(train_df["project_id"].unique()))
print("Val-Projekte:  ", sorted(val_df["project_id"].unique()))
print("Test (held-out):", sorted(test_df["project_id"].unique()))
print("n train / val / test:", len(train_df), len(val_df), len(test_df))
print("Klassenverteilung train (0 = positiv, 1 = negativ):")
print(train_df[TARGET].value_counts().sort_index())
print("Klassenverteilung val (0 = positiv, 1 = negativ):")
print(val_df[TARGET].value_counts().sort_index())

KPI-CSV all_projects_kpis_20260917_103958.csv
Train-Projekte: ['P01', 'P02', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11']
Val-Projekte:   ['P01', 'P02', 'P04', 'P05', 'P06', 'P07', 'P08', 'P10']
Test (held-out): ['P12', 'P13']
n train / val / test: 380 130 230
Klassenverteilung train (0 = positiv, 1 = negativ):
is_neg
0    300
1     80
Name: count, dtype: int64
Klassenverteilung val (0 = positiv, 1 = negativ):
is_neg
0    105
1     25
Name: count, dtype: int64


4) Preprocessing

- Fehlende Kpi werte ersetzen über den median
- Numerische Kpis standardisieren -> Wichtig für die Logistische Regression
- Die Umgebung (env) wird one-hot encodiert

Der Preprocessor wird ausschließlich auf die Trainingsdaten trainiert und dann auf die Validierungs und Evaluierungsdaten angewandt.

Untersuchungsschritt: Im endgültigen Modell (Abschnitt 11) entfällt die Umgebungs-Kodierung, weil sie messbar nichts beiträgt. Die Imputation, die Logarithmierung und die Skalierung bleiben.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

log_pipeline = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")), 
        ("log",FunctionTransformer (np.log1p, feature_names_out = "one-to-one")),
        ("scaler", StandardScaler()),
    ]
)

plain_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop=["low"])),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_pipeline, LOG_FEATURES),
        ("plain", plain_pipeline, PLAIN_FEATURES),
        ("cat", categorical_pipeline, CAT_FEATURES),
    ],
    sparse_threshold=0.0,
    verbose_feature_names_out=False,
)

X_train = preprocessor.fit_transform(
    train_df[LOG_FEATURES + PLAIN_FEATURES + CAT_FEATURES]
)
X_val = preprocessor.transform(val_df[LOG_FEATURES + PLAIN_FEATURES + CAT_FEATURES])
y_train = train_df[TARGET]
y_val = val_df[TARGET]

print("Erzeugte Feature-Spalten:", list(preprocessor.get_feature_names_out()))
print("X_train:", X_train.shape, "| X_val:", X_val.shape)
print(
    "Fehlende Werte train / val:",
    int(np.isnan(X_train).sum()),
    int(np.isnan(X_val).sum()),
)
print("Klassen train (0 / 1):", int((y_train == 0).sum()), int((y_train == 1).sum()))
print("Klassen val   (0 / 1):", int((y_val == 0).sum()), int((y_val == 1).sum()))

Erzeugte Feature-Spalten: ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec', 'error_rate_percent', 'env_extreme', 'env_high', 'env_medium', 'env_prod']
X_train: (380, 8) | X_val: (130, 8)
Fehlende Werte train / val: 0 0
Klassen train (0 / 1): 300 80
Klassen val   (0 / 1): 105 25


5. Modelle und Kennzahlen

- Logistische Regression in zwei Gewichtungen
- Dummy-Klassifikator dient als Vergleich
- Kennzahlen: PR-AUC, F1, Precision, Recall, Confusion Matrix

Untersuchungsschritt: Hier entsteht die erste Vergleichslinie überhaupt. Das endgültige Modell steht in Abschnitt 11.

In [8]:
modelle = {
    "LogReg (keine Gewichtung)": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "LogReg (balanced)": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Dummy (most_frequent)": DummyClassifier(strategy="most_frequent"),
}

ergebnisse = []

for name, klassifikator in modelle.items():
    klassifikator.fit(X_train, y_train)
    y_pred = klassifikator.predict(X_val)
    y_score = klassifikator.predict_proba(X_val)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    ergebnisse.append(
        {
            "Modell": name,
            "PR-AUC": round(average_precision_score(y_val, y_score), 3),
            "F1": round(f1_score(y_val, y_pred, zero_division=0), 3),
            "Precision": round(precision_score(y_val, y_pred, zero_division=0), 3),
            "Recall": round(recall_score(y_val, y_pred, zero_division=0), 3),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        }
    )

ergebnis_df = pd.DataFrame(ergebnisse).set_index("Modell")

print("Zufalls-Baseline PR-AUC (Anteil der Negativen in val):", round(y_val.mean(), 3))
print()
print(ergebnis_df)

Zufalls-Baseline PR-AUC (Anteil der Negativen in val): 0.192

                           PR-AUC     F1  Precision  Recall   TN  FP  FN  TP
Modell                                                                      
LogReg (keine Gewichtung)   0.616  0.077      1.000    0.04  105   0  24   1
LogReg (balanced)           0.622  0.454      0.306    0.88   55  50   3  22
Dummy (most_frequent)       0.192  0.000      0.000    0.00  105   0  25   0


6. Entscheidungsgrenze und Fehleranalyse

- Der Standardwert 0,5 ist bei 19 % Negativanteil unbrauchbar (Recall 0,04)
-  Die Grenze wird aus ehrlichen Trainingsdaten bestimmt (Out-of-Fold über GroupKFold), nicht auf der Validierung
-  Recall-Ziel 0,8: eine übersehene Mutation (FN) ist teurer als ein Fehlalarm (FP)
-  Die Grenze wird anschließend nur einmal auf die Validierung angewandt (kein Nachjustieren)
-  Die Trennschärfe ändert sich dadurch nicht: die PR-AUC bleibt unverändert
-  Fehleranalyse: welche Mutationen übersehen wurden und wo die Fehlalarme sitzen (je Umgebungsstufe)

In [9]:
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import GroupKFold, cross_val_predict

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

scores_train = cross_val_predict(
    logreg,
    X_train,
    y_train,
    cv=GroupKFold(n_splits=5),
    groups=train_df["pair_id"],
    method="predict_proba",
)[:, 1]

precision, recall, schwellen = precision_recall_curve(y_train, scores_train)

ziel_recall = 0.8
geeignet = recall[:-1] >= ziel_recall
SCHWELLE = schwellen[geeignet].max()

logreg.fit(X_train, y_train)
y_score_val = logreg.predict_proba(X_val)[:, 1]
y_pred_val = (y_score_val >= SCHWELLE).astype(int)

tn, fp, fn, tp = confusion_matrix(y_val, y_pred_val).ravel()

print("PR-AUC train (Out-of-Fold):", round(average_precision_score(y_train, scores_train), 3))
print("Zufalls-Baseline train:", round(y_train.mean(), 3))
print("Gewählter Schwellenwert:", round(SCHWELLE, 3), "für Recall-Ziel", ziel_recall)
print()
print("PR-AUC val:", round(average_precision_score(y_val, y_score_val), 3))
print("F1:", round(f1_score(y_val, y_pred_val, zero_division=0), 3))
print("Precision:", round(precision_score(y_val, y_pred_val, zero_division=0), 3))
print("Recall:", round(recall_score(y_val, y_pred_val, zero_division=0), 3))
print()
print("Mutation erkannt (TP):", tp)
print("Mutation übersehen (FN):", fn)
print("Fehlalarm (FP):", fp)
print("korrekt freigegeben (TN):", tn)

PR-AUC train (Out-of-Fold): 0.377
Zufalls-Baseline train: 0.211
Gewählter Schwellenwert: 0.184 für Recall-Ziel 0.8

PR-AUC val: 0.616
F1: 0.436
Precision: 0.289
Recall: 0.88

Mutation erkannt (TP): 22
Mutation übersehen (FN): 3
Fehlalarm (FP): 54
korrekt freigegeben (TN): 51


6b) Fehleranalyse

- Die Kennzahlen verraten nur das etwas nicht passt. Eine genauere Fehleranalyse findet nun statt.

Was ich hier wissen will:

- welche Mutationen wurden übersehen?
- wie gut werden die Fehler je Umgebung erkannt? Greift das Modell unter Last besser?
- wo sitzen die Fehlalarme? Fallen sie vor allem in high und extreme?

Der letzte Punkt ist der wichtigste. Wenn die Fehlalarme fast nur in high und extreme hängen, dann reagiert das Modell einfach auf die Last und nicht auf die Mutation. Merkmale werden dann umgestellt auf lastunabhängigere(Verhältnis zum Positiv-Snippet derselben Umgebung). 

Nebenbei schaue ich mir noch an, welche Methoden überhaupt in der Validierung liegen. Das erklärt hoffentlich, warum die Out-of-Fold-Zahl (0,377) so viel schlechter aussieht als die Validierungszahl (0,616) 
- nämlich dann, wenn in der Validierung vor allem die gut messbaren Fälle gelandet sind.

In [10]:
fehler = val_df[["project_id", "target_method", "env", TARGET]].copy()
fehler["score"] = y_score_val
fehler["vorhersage"] = y_pred_val

print("Negativ-Methoden in der Validierung:")
print(sorted(val_df.loc[val_df[TARGET] == 1, "target_method"].unique()))
print()

uebersehen = fehler[(fehler[TARGET] == 1) & (fehler["vorhersage"] == 0)]
print("Übersehene Mutationen (FN):", len(uebersehen))
print(uebersehen[["target_method", "env", "score"]].sort_values("score", ascending=False))
print()

mutationen = fehler[fehler[TARGET] == 1].groupby("env").size().rename("mutationen")
erkannt = (
    fehler[(fehler[TARGET] == 1) & (fehler["vorhersage"] == 1)]
    .groupby("env")
    .size()
    .rename("erkannt")
)
trefferquote = mutationen.to_frame().join(erkannt).fillna(0).astype(int)
trefferquote["trefferquote"] = (trefferquote["erkannt"] / trefferquote["mutationen"]).round(3)
print("Erkennungsquote je Umgebung:")
print(trefferquote)
print()

gesunde = fehler[fehler[TARGET] == 0].groupby("env").size().rename("gesunde_laeufe")
alarme = (
    fehler[(fehler[TARGET] == 0) & (fehler["vorhersage"] == 1)]
    .groupby("env")
    .size()
    .rename("fehlalarme")
)
fehlalarmquote = gesunde.to_frame().join(alarme).fillna(0).astype(int)
fehlalarmquote["fehlalarmquote"] = (
    fehlalarmquote["fehlalarme"] / fehlalarmquote["gesunde_laeufe"]
).round(3)
print("Fehlalarmquote je Umgebung:")
print(fehlalarmquote)

Negativ-Methoden in der Validierung:
['pos_010_post_sign_in_user', 'pos_011_post_create_user', 'pos_032_login', 'pos_035_list_users', 'pos_079_manager_authenticate']

Übersehene Mutationen (FN): 3
                target_method      env     score
263        pos_035_list_users  extreme  0.134253
118  pos_011_post_create_user  extreme  0.076970
117  pos_011_post_create_user     high  0.072294

Erkennungsquote je Umgebung:
         mutationen  erkannt  trefferquote
env                                       
extreme           5        3           0.6
high              5        4           0.8
low               5        5           1.0
medium            5        5           1.0
prod              5        5           1.0

Fehlalarmquote je Umgebung:
         gesunde_laeufe  fehlalarme  fehlalarmquote
env                                                
extreme              21          11           0.524
high                 21          12           0.571
low                  21          10    

7. Ehrliche Auswertung und Label-Qualität

Die 0,616 aus der Validierung ist nicht belastbar, weil dort zufällig nur die fünf starken Methoden gelandet sind. Also rechne ich jetzt über alle 510 Zeilen: fünf Gruppierungen (GroupKFold über pair_id), jedes Modell sieht vier Fünftel der Methoden und bewertet das übrige Fünftel. So bekommt jede Zeile eine Vorhersage von einem Modell, das sie nicht kennt, und ich sehe zusätzlich die Streuung zwischen den Gruppen.

Danach derselbe Durchlauf ohne die fünf Grenzfälle (029, 041, 046, 056, 063). Diese Methoden waren in den Rohdaten nicht vom gesunden Verhalten zu unterscheiden - das ist keine Modellschwäche, sondern eine Grenze der Messung. Ich lasse sie deshalb aus den Negativ-Beispielen weg und stelle beide Varianten gegenüber. Das Ausschlusskriterium ist der Kampagnen-Befund von vor der Modellierung und nicht der Modellfehler, sonst wäre es Cherry-Picking.

Wichtig: Preprocessing und Modell stecken jetzt in einer Pipeline und werden pro Gruppierung neu angepasst. Nur so stammen Median, Skalierung und Kategorien ausschließlich aus den Trainingszeilen der jeweiligen Gruppierung.

In [11]:
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve

GRENZFAELLE = ["_029_", "_041_", "_046_", "_056_", "_063_"]

spalten = LOG_FEATURES + PLAIN_FEATURES + CAT_FEATURES


def basis_modell():
    return Pipeline(
        steps=[
            ("prep", clone(preprocessor)),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )


def auswerten(daten, beschriftung, modell_bauen=basis_modell):
    X = daten[spalten]
    y = daten[TARGET].to_numpy()
    scores = np.zeros(len(daten))
    ap_pro_fold = []

    for train_teil, test_teil in GroupKFold(n_splits=5).split(
        X, y, groups=daten["pair_id"]
    ):
        modell = modell_bauen()
        modell.fit(X.iloc[train_teil], y[train_teil])
        teil_scores = modell.predict_proba(X.iloc[test_teil])[:, 1]
        scores[test_teil] = teil_scores
        ap_pro_fold.append(average_precision_score(y[test_teil], teil_scores))

    precision, recall, schwellen = precision_recall_curve(y, scores)
    geeignet = recall[:-1] >= 0.8
    schwelle = schwellen[geeignet].max()
    vorhersage = (scores >= schwelle).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, vorhersage).ravel()

    print(beschriftung)
    print("Zeilen:", len(daten), "| Negative:", int(y.sum()), "| Zufalls-AP:", round(y.mean(), 3))
    print(
        "AP je Gruppe:",
        [round(wert, 3) for wert in ap_pro_fold],
        "| Mittel:",
        round(np.mean(ap_pro_fold), 3),
        "+/-",
        round(np.std(ap_pro_fold), 3),
    )
    print("AP gepoolt (Out-of-Fold):", round(average_precision_score(y, scores), 3))
    print(
        "bei Schwelle",
        round(schwelle, 3),
        "-> Precision:",
        round(precision_score(y, vorhersage, zero_division=0), 3),
        "| Recall:",
        round(recall_score(y, vorhersage, zero_division=0), 3),
        "| F1:",
        round(f1_score(y, vorhersage, zero_division=0), 3),
    )
    print("TP/FP/FN/TN:", tp, fp, fn, tn)
    print()

    return {
        "Variante": beschriftung,
        "AP gepoolt": round(average_precision_score(y, scores), 3),
        "AP Mittel": round(np.mean(ap_pro_fold), 3),
        "AP Streuung": round(np.std(ap_pro_fold), 3),
    }


ohne_grenzfaelle = train_val[
    ~(
        (train_val[TARGET] == 1)
        & train_val["target_method"].str.contains("|".join(GRENZFAELLE))
    )
]

ergebnis_df = pd.DataFrame(
    [
        auswerten(train_val, "Basis, alle 21 Negativ-Methoden"),
        auswerten(ohne_grenzfaelle, "Basis, ohne die fünf Grenzfälle"),
    ]
).set_index("Variante")

print(ergebnis_df)

Basis, alle 21 Negativ-Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.45, 0.432, 0.43, 0.723, 0.534] | Mittel: 0.514 +/- 0.111
AP gepoolt (Out-of-Fold): 0.425
bei Schwelle 0.154 -> Precision: 0.318 | Recall: 0.8 | F1: 0.455
TP/FP/FN/TN: 84 180 21 225

Basis, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.44, 0.47, 0.439, 0.719, 0.53] | Mittel: 0.52 +/- 0.105
AP gepoolt (Out-of-Fold): 0.412
bei Schwelle 0.1 -> Precision: 0.259 | Recall: 0.8 | F1: 0.391
TP/FP/FN/TN: 64 183 16 222

                                 AP gepoolt  AP Mittel  AP Streuung
Variante                                                           
Basis, alle 21 Negativ-Methoden       0.425      0.514        0.111
Basis, ohne die fünf Grenzfälle       0.412      0.520        0.105


8. Unterschiedliche Steigungen je Laststufe

Die Fehleranalyse hat gezeigt, dass das Modell bei high und extreme schlechter erkennt. Der Grund steckt im Aufbau des Modells: Mit den Umgebungs-Spalten kann es nur das Niveau je Laststufe verschieben, nicht die Steigung der KPI. Ein Effekt, der bei geringer Last kaum messbar ist und bei hoher Last deutlich ausfällt - genau das war das Ergebnis der Kampagne - lässt sich damit nicht abbilden.

Deshalb multipliziere ich hier jede KPI mit jeder Umgebungs-Spalte. Das Modell bekommt dadurch für jede Kombination einen eigenen Koeffizienten und kann pro Laststufe unterschiedlich stark auf Latenz oder Durchsatz reagieren. Die Grundmerkmale bleiben erhalten, die sechzehn Produkte kommen dazu.

Beide Konfigurationen - mit und ohne Grenzfälle - laufen durch dieselbe Auswertung wie in der Zelle davor, damit die Zahlen direkt vergleichbar sind.

In [12]:
from sklearn.preprocessing import FunctionTransformer

ANZAHL_KPI = len(LOG_FEATURES) + len(PLAIN_FEATURES)


def mit_interaktionen(X):
    kpi = X[:, :ANZAHL_KPI]
    umgebung = X[:, ANZAHL_KPI:]
    produkte = (kpi[:, :, None] * umgebung[:, None, :]).reshape(X.shape[0], -1)
    return np.hstack([X, produkte])


def interaktions_modell():
    return Pipeline(
        steps=[
            ("prep", clone(preprocessor)),
            ("inter", FunctionTransformer(mit_interaktionen)),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )


vergleich_df = pd.DataFrame(
    [
        auswerten(train_val, "Basis, alle 21 Negativ-Methoden"),
        auswerten(ohne_grenzfaelle, "Basis, ohne die fünf Grenzfälle"),
        auswerten(
            train_val,
            "Mit Interaktionen, alle 21 Negativ-Methoden",
            modell_bauen=interaktions_modell,
        ),
        auswerten(
            ohne_grenzfaelle,
            "Mit Interaktionen, ohne die fünf Grenzfälle",
            modell_bauen=interaktions_modell,
        ),
    ]
).set_index("Variante")

print(vergleich_df)

Basis, alle 21 Negativ-Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.45, 0.432, 0.43, 0.723, 0.534] | Mittel: 0.514 +/- 0.111
AP gepoolt (Out-of-Fold): 0.425
bei Schwelle 0.154 -> Precision: 0.318 | Recall: 0.8 | F1: 0.455
TP/FP/FN/TN: 84 180 21 225

Basis, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.44, 0.47, 0.439, 0.719, 0.53] | Mittel: 0.52 +/- 0.105
AP gepoolt (Out-of-Fold): 0.412
bei Schwelle 0.1 -> Precision: 0.259 | Recall: 0.8 | F1: 0.391
TP/FP/FN/TN: 64 183 16 222

Mit Interaktionen, alle 21 Negativ-Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.451, 0.404, 0.301, 0.705, 0.523] | Mittel: 0.477 +/- 0.135
AP gepoolt (Out-of-Fold): 0.368
bei Schwelle 0.155 -> Precision: 0.316 | Recall: 0.8 | F1: 0.453
TP/FP/FN/TN: 84 182 21 223

Mit Interaktionen, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.439, 0.436, 0.289, 0.689, 0.573] | Mittel: 0.4

9. Zweites Modell: LightGBM

Die Logistische Regression zieht eine gerade Trennlinie zwischen gesund und mutiert. Bei Latenzwerten ist das eine starke Vereinfachung: Ein Anstieg von 300 auf 600 Millisekunden wirkt dort anders als einer von 2000 auf 4000. Die Interaktionsmerkmale sollten das auffangen und haben es schlechter gemacht, weil aus 8 Merkmalen auf 380 Zeilen gleich 24 wurden.

Deshalb probiere ich jetzt ein Modell, das Schwellen und Krümmungen selbst lernt: LightGBM mit bewusst kleinen Bäumen. Die Bäume bleiben flach (höchstens sieben Blätter), jedes Blatt braucht mindestens zwanzig Zeilen, und die Regularisierung ist aktiv. Ohne diese Bremsen merkt sich der Baumbau bei dieser Datenmenge einzelne Zeilen und liefert auf neuen Daten nichts mehr.

Das Preprocessing bleibt unverändert im Modell, obwohl Bäume keine Skalierung brauchen. So unterscheiden sich die beiden Läufe nur im Klassifikator, und ich vergleiche tatsächlich zwei Modelle und nicht zwei Vorverarbeitungen.

Wichtig: P12 und P13 werden hier nicht angefasst. Alle Zahlen kommen weiterhin aus der gruppierten Kreuzvalidierung über P01 bis P11, die unabhängige Prüfung bleibt für den Schluss reserviert.

Untersuchungsschritt: Die hier gewählte Merkmalsmenge (vier KPI plus Umgebung) ist nicht der Endstand. Das endgültige Modell steht in Abschnitt 11.

In [13]:
from lightgbm import LGBMClassifier


def lightgbm_modell():
    return Pipeline(
        steps=[
            ("prep", clone(preprocessor)),
            (
                "clf",
                LGBMClassifier(
                    n_estimators=300,
                    learning_rate=0.05,
                    num_leaves=7,
                    min_child_samples=20,
                    subsample=0.8,
                    subsample_freq=1,
                    colsample_bytree=0.8,
                    reg_lambda=1.0,
                    random_state=RANDOM_STATE,
                    verbose=-1,
                ),
            ),
        ]
    )


vergleich_lgbm = pd.DataFrame(
    [
        auswerten(train_val, "LogReg, alle 21 Methoden"),
        auswerten(ohne_grenzfaelle, "LogReg, ohne die fünf Grenzfälle"),
        auswerten(train_val, "LightGBM, alle 21 Methoden", modell_bauen=lightgbm_modell),
        auswerten(
            ohne_grenzfaelle,
            "LightGBM, ohne die fünf Grenzfälle",
            modell_bauen=lightgbm_modell,
        ),
    ]
).set_index("Variante")

print(vergleich_lgbm)

wichtigkeits_modell = lightgbm_modell()
wichtigkeits_modell.fit(train_val[spalten], train_val[TARGET])
namen = wichtigkeits_modell.named_steps["prep"].get_feature_names_out()
bedeutung = wichtigkeits_modell.named_steps["clf"].feature_importances_

print()
print("Feature-Bedeutung LightGBM (alle Trainingszeilen):")
print(pd.Series(bedeutung, index=namen).sort_values(ascending=False))

LogReg, alle 21 Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.45, 0.432, 0.43, 0.723, 0.534] | Mittel: 0.514 +/- 0.111
AP gepoolt (Out-of-Fold): 0.425
bei Schwelle 0.154 -> Precision: 0.318 | Recall: 0.8 | F1: 0.455
TP/FP/FN/TN: 84 180 21 225

LogReg, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.44, 0.47, 0.439, 0.719, 0.53] | Mittel: 0.52 +/- 0.105
AP gepoolt (Out-of-Fold): 0.412
bei Schwelle 0.1 -> Precision: 0.259 | Recall: 0.8 | F1: 0.391
TP/FP/FN/TN: 64 183 16 222

LightGBM, alle 21 Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.655, 0.537, 0.634, 0.943, 0.541] | Mittel: 0.662 +/- 0.148
AP gepoolt (Out-of-Fold): 0.652
bei Schwelle 0.068 -> Precision: 0.373 | Recall: 0.8 | F1: 0.509
TP/FP/FN/TN: 84 141 21 264

LightGBM, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.584, 0.712, 0.741, 0.861, 0.691] | Mittel: 0.718 +/- 0.089
AP gepoolt (Out-of-

10. SHAP: welches Merkmal zieht in welche Richtung?

Die Feature-Bedeutung von LightGBM sagt mir nur, welches Merkmal oft zum Einsatz kommt - nicht, ab welchem Wert es die Bewertung kippt. SHAP beantwortet das für dieses Modell: in welcher Richtung ein Merkmal wirkt und ab welcher Größenordnung sich das Vorzeichen dreht.

Erklärt wird hier das Modell aus Abschnitt 9, also noch mit vier Kennzahlen und der Umgebungsstufe. Das endgültige Modell mit drei Merkmalen entsteht erst in Abschnitt 11. Die Aussagen über Richtung und Größenordnung gelten für die drei Größen, die beide Modelle teilen, und bleiben deshalb gültig.

Nicht abgeleitet werden daraus die Schwellen der Bewertungsmatrix. Deren Bänder sind an den eigenen Messdaten kalibriert (Durchsatzverlust 10, 30, 60 Prozent; Latenzanstieg 20, 40, 60 Prozent; Mindestabweichung 3 Millisekunden) und sie sind relativ - Abweichung zur Positivmessung derselben Methode und Umgebungsstufe. SHAP liefert dagegen absolute Beiträge in Millisekunden und Anfragen pro Sekunde. Die beiden Skalen sind nicht ineinander übertragbar, und aus der Richtung eines Merkmals folgt keine Bandgrenze.

Was die Auswertung hier leistet: Sie zeigt, welches Merkmal den Ausschlag gibt und ab welcher Größenordnung, und sie macht damit die Wirkung des Modells nachvollziehbar. Das ist die Grundlage für den Ausblick, das Modell später im referenzlosen Fall einzusetzen, und die Voraussetzung dafür, dass auch eine Modellausgabe begründet werden kann. Für die Erklärbarkeit der Matrix ist sie nicht nötig: Deren Ausgabe trägt Kriterium, Abweichung, Gewicht und Hauptursache bereits mit sich.

SHAP zerlegt jede einzelne Vorhersage in Beiträge pro Merkmal. Positiv heißt: treibt die Vorhersage in Richtung des Etiketts "mutiert", negativ heißt: spricht für gesund. Weil das Modell auf den logarithmierten und standardisierten Werten rechnet, sortiere ich die Beiträge anschließend wieder nach den Originalwerten - so stehen in der Ausgabe Millisekunden und Anfragen pro Sekunde, nicht Standardabweichungen.

Zwei Auswertungen: eine Rangfolge nach mittlerem Betrag der Beiträge, und je Merkmal fünf gleich große Wertebereiche mit dem durchschnittlichen Beitrag. Diese fünf Bereiche zeigen, wo das Vorzeichen umschlägt - also den Bereich, ab dem das Merkmal in Richtung auffällig zieht.

Wichtig: Das Modell wird hier auf allen Trainingszeilen angelernt und nur erklärt, nicht bewertet. Die Zahlen aus der Kreuzvalidierung bleiben unberührt.

In [14]:
import shap

daten_shap = ohne_grenzfaelle.copy()
modell_shap = lightgbm_modell()
modell_shap.fit(daten_shap[spalten], daten_shap[TARGET])

vorbereitet = modell_shap.named_steps["prep"].transform(daten_shap[spalten])
namen = modell_shap.named_steps["prep"].get_feature_names_out()

erklaerer = shap.TreeExplainer(modell_shap.named_steps["clf"])
werte = erklaerer.shap_values(vorbereitet)
if isinstance(werte, list):
    werte = werte[1]
elif getattr(werte, "ndim", 2) == 3:
    werte = werte[:, :, 1]

shap_df = pd.DataFrame(werte, columns=namen)
original = daten_shap[list(namen[:4])].reset_index(drop=True)

print("Mittlerer Betrag der SHAP-Beiträge:")
print(shap_df.abs().mean().sort_values(ascending=False).round(3))
print()

for merkmal in ["requests_per_sec", "p95_latency_ms", "avg_latency_ms"]:
    bereiche = pd.qcut(original[merkmal], 5, duplicates="drop")
    uebersicht = pd.DataFrame(
        {
            "wert_von": original[merkmal].groupby(bereiche, observed=True).min().round(1),
            "wert_bis": original[merkmal].groupby(bereiche, observed=True).max().round(1),
            "shap_mittel": shap_df[merkmal].groupby(bereiche, observed=True).mean().round(3),
        }
    )
    print(merkmal)
    print(uebersicht)
    print()

/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mittlerer Betrag der SHAP-Beiträge:
requests_per_sec      1.577
error_rate_percent    0.685
p95_latency_ms        0.639
avg_latency_ms        0.585
env_extreme           0.068
env_prod              0.053
env_high              0.030
env_medium            0.014
dtype: float64

requests_per_sec
                  wert_von  wert_bis  shap_mittel
requests_per_sec                                 
(0.339, 3.35]          0.3       3.4        1.820
(3.35, 6.96]           3.4       7.0       -1.430
(6.96, 40.632]         7.0      40.6        1.099
(40.632, 85.708]      40.6      85.6       -0.167
(85.708, 234.94]      86.0     234.9       -1.496

p95_latency_ms
                    wert_von  wert_bis  shap_mittel
p95_latency_ms                                     
(6.999, 248.0]             7       240       -0.269
(248.0, 500.0]           250       500       -0.724
(500.0, 3340.0]          510      3300       -0.310
(3340.0, 16000.0]       3400     16000        0.427
(16000.0, 79000.0]     17000 

/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/.venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


11.  Variante A: Festlegung des Endmodells und Beleg auf den Trainingsprojekten

Die Untersuchung ist abgeschlossen, hier wird das Modell festgelegt. Gewählt ist ein **LightGBM** (`lgbm_klassifikator`, in der Zelle unten). Es nutzt drei Merkmale: mittlere Latenz, p95-Latenz und Durchsatz. Keine Umgebungsmerkmale, weil sie messbar nichts beitragen (p = 0,99). Keine Fehlerquote, weil sie in diesen Daten kein Mutationssignal ist, sondern ein Lastsignal: Sie steigt in den gesunden Referenzläufen unter hoher Last, trennt die Klassen in der Randverteilung nicht und wirkt im Modell in umgekehrter Richtung als fachlich erwartet. Der Preis des Verzichts ist beziffert und klein.

Zur Kontrolle läuft die Variante mit Fehlerquote daneben, damit der Unterschied sichtbar bleibt und nicht der Eindruck entsteht, ein wirksames Merkmal sei ohne Grund entfernt worden.

Bewertet wird über zwei Wege. Zehn gruppierte Aufteilungen liefern die mittlere AP mit ihrer Streuung, weil eine einzelne Aufteilung bei dieser Datenmenge um den Faktor sieben schwanken kann. Die fünffache gruppierte Kreuzvalidierung liefert zusätzlich Precision, Recall und die Verwirrungsmatrix bei der Entscheidungsgrenze, die aus den Trainingsdaten für eine Trefferquote von 80 Prozent bestimmt wird - also ohne Blick auf die Prüfzeilen.

Gerechnet wird auf P01 bis P11 ohne die fünf Grenzfälle (029, 041, 046, 056, 063). Diese Methoden waren in den Rohdaten nicht vom gesunden Verhalten zu unterscheiden; dass sie ausgeschlossen sind, ist der Kampagnen-Befund von vor der Modellierung, nicht das Ergebnis der Modellfehler.

Alle Zellen davor sind Untersuchungsschritte: Sie zeigen, wie es zu dieser Wahl kam (lineares Modell, Interaktionsmerkmale, Merkmalsprüfung, Modellvergleich). Gültig für die Arbeit ist das Modell aus diesem Abschnitt.

**Bezeichnungen ab hier.** Damit beim Weiterlesen eindeutig ist, wovon die Rede ist:

- **Variante A** = dieses Modell, also **LightGBM auf den Messwerten** (mittlere Latenz, p95-Latenz, Durchsatz). Angewandt auf das Evaluationsdatenset in Abschnitt 15.
- **Variante B** = das Modell aus Abschnitt 13, eine **logistische Regression auf Zeichenfolgen des Snippet-Textes**. Sie sieht ausschließlich Code, keine Messwerte.
- **Variante C** = Abschnitt 14, Code plus die Messwerte der leichtesten Stufe (low).
- Die Abschnitte 5 bis 10 sind **Untersuchungsschritte** – logistische Regression, Interaktionsmerkmale, Modellvergleich, SHAP – und tragen keine dieser Bezeichnungen.


In [15]:
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)

FEATURES_FINAL = ["avg_latency_ms", "p95_latency_ms", "requests_per_sec"]
FEATURES_REFERENZ = FEATURES_FINAL + ["error_rate_percent"]


def lgbm_klassifikator():
    return LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=7,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        verbose=-1,
    )


def vorbereitung(numerische, mit_fehlerquote):
    schritte = [
        (
            "log",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    (
                        "log",
                        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                    ),
                    ("scaler", StandardScaler()),
                ]
            ),
            numerische,
        )
    ]
    if mit_fehlerquote:
        schritte.append(
            (
                "plain",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                ["error_rate_percent"],
            )
        )
    return ColumnTransformer(
        transformers=schritte, sparse_threshold=0.0, verbose_feature_names_out=False
    )


def modell_bauen(numerische, mit_fehlerquote):
    return Pipeline(
        steps=[
            ("prep", vorbereitung(numerische, mit_fehlerquote)),
            ("clf", lgbm_klassifikator()),
        ]
    )


def pruefe(daten, features, mit_fehlerquote, beschriftung):
    X = daten[features]
    y = daten[TARGET].to_numpy()
    gruppen = daten["pair_id"].to_numpy()

    werte = []
    for seed in range(1, 11):
        train_teil, test_teil = next(
            GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
                X, y, groups=gruppen
            )
        )
        modell = modell_bauen(features, mit_fehlerquote)
        modell.fit(X.iloc[train_teil], y[train_teil])
        scores = modell.predict_proba(X.iloc[test_teil])[:, 1]
        werte.append(average_precision_score(y[test_teil], scores))

    scores_kreuz = np.zeros(len(daten))
    for train_teil, test_teil in GroupKFold(n_splits=5).split(X, y, groups=gruppen):
        modell = modell_bauen(features, mit_fehlerquote)
        modell.fit(X.iloc[train_teil], y[train_teil])
        scores_kreuz[test_teil] = modell.predict_proba(X.iloc[test_teil])[:, 1]

    precision, recall, schwellen = precision_recall_curve(y, scores_kreuz)
    schwelle = schwellen[recall[:-1] >= 0.8].max()
    vorhersage = (scores_kreuz >= schwelle).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, vorhersage).ravel()

    print(beschriftung)
    print("  Merkmale:", features)
    print(
        "  AP aus zehn Aufteilungen:",
        round(np.mean(werte), 3),
        "+/-",
        round(np.std(werte), 3),
    )
    print(
        "  AP aus fünffacher Kreuzvalidierung:",
        round(average_precision_score(y, scores_kreuz), 3),
    )
    print(
        "  Zufallswert:",
        round(y.mean(), 3),
        "| Entscheidungsgrenze:",
        round(schwelle, 3),
    )
    print(
        "  Precision:",
        round(precision_score(y, vorhersage, zero_division=0), 3),
        "| Recall:",
        round(recall_score(y, vorhersage, zero_division=0), 3),
        "| F1:",
        round(f1_score(y, vorhersage, zero_division=0), 3),
    )
    print("  Verwirrungsmatrix  TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)
    print()

    return {
        "Variante": beschriftung,
        "AP 10 Aufteilungen": round(np.mean(werte), 3),
        "AP Streuung": round(np.std(werte), 3),
        "AP Kreuzvalidierung": round(average_precision_score(y, scores_kreuz), 3),
        "Precision": round(precision_score(y, vorhersage, zero_division=0), 3),
        "Recall": round(recall_score(y, vorhersage, zero_division=0), 3),
        "F1": round(f1_score(y, vorhersage, zero_division=0), 3),
        "FP": fp,
        "FN": fn,
    }


abschluss_df = pd.DataFrame(
    [
        pruefe(ohne_grenzfaelle, FEATURES_FINAL, False, "Endmodell (3 Merkmale)"),
        pruefe(
            ohne_grenzfaelle,
            FEATURES_FINAL + ["error_rate_percent"],
            True,
            "Referenz (mit Fehlerquote)",
        ),
    ]
).set_index("Variante")

print(abschluss_df)

Endmodell (3 Merkmale)
  Merkmale: ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec']
  AP aus zehn Aufteilungen: 0.526 +/- 0.178
  AP aus fünffacher Kreuzvalidierung: 0.638
  Zufallswert: 0.165 | Entscheidungsgrenze: 0.027
  Precision: 0.282 | Recall: 0.8 | F1: 0.417
  Verwirrungsmatrix  TN: 242 FP: 163 FN: 16 TP: 64

Referenz (mit Fehlerquote)
  Merkmale: ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec', 'error_rate_percent']
  AP aus zehn Aufteilungen: 0.556 +/- 0.179
  AP aus fünffacher Kreuzvalidierung: 0.69
  Zufallswert: 0.165 | Entscheidungsgrenze: 0.042
  Precision: 0.33 | Recall: 0.8 | F1: 0.467
  Verwirrungsmatrix  TN: 275 FP: 130 FN: 16 TP: 64

                            AP 10 Aufteilungen  AP Streuung  \
Variante                                                      
Endmodell (3 Merkmale)                   0.526        0.178   
Referenz (mit Fehlerquote)               0.556        0.179   

                            AP Kreuzvalidierung  Precision  Recall   

Erstellung Der Datensets 

12 A Etikettdatensatz mit Snippets – Trainingszeilen P01–P11

Hier entsteht der Trainingsdatensatz für den Teil, der die eigentliche Vorhersage macht: Aus dem Code soll abgeleitet werden, wie sich die Messwerte verhalten werden - bevor gemessen wurde.

Das Etikett kommt nicht aus den Modellfehlern, sondern aus der Bewertungsmatrix. Für jede Zeile wird über den Prototypen der Punktwert berechnet: Abweichung zur Referenz derselben Methode und Umgebungsstufe, Bandzuordnung, gewichtete Summe. Daraus entstehen zwei Ziele. Das erste ist die Einstufung in stabil, Grenzfall oder instabil. Das zweite ist das Hauptkriterium - also die Frage, welcher der drei Werte die Bewertung nach unten gezogen hat; dieses Ziel ist die interessantere Aussage, weil der Agent später sagen soll, welche Kennzahl leiden wird.

Genutzt wird der Analysesatz: P01 bis P11, ohne das ausgeschlossene Projekt P03 und ohne die fünf nicht trennscharfen Mutationen. Das sind 485 Zeilen, davon 405 gesund und 80 mutiert, alle mit Snippet-Code.

Eine Einschränkung, die in die Arbeit gehört: Gesunde Läufe bekommen den Punktwert 100, weil sie mit ihrer eigenen Referenz verglichen werden - sie können per Konstruktion keine Abweichung haben. Das Etikett ist dadurch deutlich schiefer als das ursprüngliche KPI-Label: Nur die Mutationen verteilen sich auf die drei Einstufungen, und ein Teil von ihnen erhält sogar stabil, weil die Matrix ihre Wirkung nicht erkennt. Genau diese Zeilen sind für das Code-Modell nicht lernbar und müssen bei der Bewertung als Grenze benannt werden.

In [16]:
import json
import sys

sys.path.insert(0, str(ROOT / "prototype"))
from scoring import bewerte, lade_hilfsdaten

referenz, notgrenzen = lade_hilfsdaten()

GRENZFAELLE_NUMMERN = {"029", "041", "046", "056", "063"}
KURZNAMEN = {
    "Durchsatz gegenüber Referenz": "durchsatz",
    "p95-Latenz gegenüber Referenz": "p95_latenz",
    "Mittlere Latenz gegenüber Referenz": "mittlere_latenz",
}

zeilen = [
    json.loads(zeile)
    for zeile in open(ROOT / "export_kpis" / "dataset.jsonl", encoding="utf-8")
]
analyse = [
    zeile
    for zeile in zeilen
    if zeile["split"] == "train"
    and not zeile["variant"].startswith("P03")
    and not (zeile["label"] == 1 and zeile["pos"] in GRENZFAELLE_NUMMERN)
]

saetze = []
for zeile in analyse:
    messung = {
        "target_method": zeile["method"],
        "env": zeile["env"],
        "variante": zeile["variant"],
        **zeile["metrics"],
    }
    antwort = bewerte(messung, referenz, notgrenzen)
    kriterien = {k["name"]: k for k in antwort["kriterien"]}
    verluste = sorted(
        (
            (k["gewicht"] * (100 - k["punkte"]), KURZNAMEN[k["name"]])
            for k in antwort["kriterien"]
        ),
        reverse=True,
    )
    saetze.append(
        {
            "variante": zeile["variant"],
            "methode": zeile["method"],
            "nummer": zeile["pos"],
            "umgebung": zeile["env"],
            "label_kpi": zeile["label"],
            "score_prozent": antwort["score_prozent"],
            "ziel_einstufung": antwort["einstufung"],
            "ziel_auffaellig": int(antwort["score_prozent"] < 75),
            "ziel_kriterium": verluste[0][1] if verluste[0][0] > 0 else "keines",
            "durchsatz_punkte": kriterien["Durchsatz gegenüber Referenz"]["punkte"],
            "p95_punkte": kriterien["p95-Latenz gegenüber Referenz"]["punkte"],
            "avg_punkte": kriterien["Mittlere Latenz gegenüber Referenz"]["punkte"],
            "durchsatz_abweichung": kriterien["Durchsatz gegenüber Referenz"]["abweichung_prozent"],
            "p95_abweichung": kriterien["p95-Latenz gegenüber Referenz"]["abweichung_prozent"],
            "avg_abweichung": kriterien["Mittlere Latenz gegenüber Referenz"]["abweichung_prozent"],
            "snippet": zeile["snippet"],
        }
    )

daten = pd.DataFrame(saetze)

print("Zeilen im Analysesatz:", len(daten), "| mit Code:", int((daten["snippet"].str.len() > 0).sum()))
print()
print("Einstufung nach KPI-Label (0 = gesund, 1 = mutiert):")
print(pd.crosstab(daten["label_kpi"], daten["ziel_einstufung"]))
print()
print("Hauptkriterium bei den Mutationen:")
print(daten[daten["label_kpi"] == 1]["ziel_kriterium"].value_counts())
print()
print("Binäres Ziel (auffällig = Score unter 75):")
print(daten["ziel_auffaellig"].value_counts().rename({0: "unauffällig", 1: "auffällig"}))
print()
print(
    "Nicht erkannte Mutationen (Etikett unauffällig, obwohl mutiert):",
    int(((daten["label_kpi"] == 1) & (daten["ziel_auffaellig"] == 0)).sum()),
)
print("Mehrheitslinie für das binäre Ziel:", round(daten["ziel_auffaellig"].mean(), 3))
print()
print("Snippet-Länge (Zeichen):")
print(daten["snippet"].str.len().describe().round(0))

ziel_datei = ROOT / "export_kpis" / "dataset_mit_labels.csv"
daten.to_csv(ziel_datei, index=False)
print()
print("Gespeichert:", ziel_datei)


Zeilen im Analysesatz: 485 | mit Code: 485

Einstufung nach KPI-Label (0 = gesund, 1 = mutiert):
ziel_einstufung  Grenzfall  instabil  stabil
label_kpi                                   
0                        0         0     405
1                        4        64      12

Hauptkriterium bei den Mutationen:
ziel_kriterium
durchsatz          40
p95_latenz         32
keines              4
mittlere_latenz     4
Name: count, dtype: int64

Binäres Ziel (auffällig = Score unter 75):
ziel_auffaellig
unauffällig    417
auffällig       68
Name: count, dtype: int64

Nicht erkannte Mutationen (Etikett unauffällig, obwohl mutiert): 12
Mehrheitslinie für das binäre Ziel: 0.14

Snippet-Länge (Zeichen):
count     485.0
mean     1109.0
std       644.0
min       198.0
25%       656.0
50%       944.0
75%      1284.0
max      3328.0
Name: snippet, dtype: float64

Gespeichert: /Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/export_kpis/dataset_mit_labels.csv


12 B Etikettdatensatz mit Snippets – Evaluationszeilen P12 und P13

Dieselbe Rechnung wie in Abschnitt 12, aber über den Evaluationssplit: P12 und P13 statt P01 bis P11. Das Etikett entsteht wie dort aus der Bewertungsmatrix – Abweichung zur Referenz derselben Methode und Umgebungsstufe, Bandzuordnung, gewichtete Summe.

Die Datei heißt `dataset_mit_labels_eval.csv` und liegt neben der Trainingsdatei. Diese bleibt unberührt, weil sämtliche Zahlen der Abschnitte 5 bis 14 auf ihr beruhen.

Warum das getrennt wird: Die Modelle sind in Abschnitt 11 und 13 festgelegt. Auf diesen Zeilen werden sie nur noch angewandt, nicht mehr ausgewählt – sonst wäre die Prüfung nicht mehr unabhängig. P12 und P13 haben bei keiner Entscheidung mitgewirkt, weder bei der Merkmalswahl noch bei den Schwellen.

Der Evaluationssatz umfasst 230 Zeilen: fünf Umgebungsstufen für P12 und P13, gesunde Läufe und Mutationen. Weil die gesunden Läufe deutlich in der Überzahl sind, ist die Prävalenz der Auffälligkeiten niedrig – und genau diese Prävalenz ist der Wert, den ein Verfahren ohne jede Modellleistung erreicht. Sie steht darum in jeder späteren Kennzahl als Vergleichsgröße daneben.

In [17]:
import json
import sys

sys.path.insert(0, str(ROOT / "prototype"))
from scoring import bewerte, lade_hilfsdaten

referenz, notgrenzen = lade_hilfsdaten()

KURZNAMEN_EVAL = {
    "Durchsatz gegenüber Referenz": "durchsatz",
    "p95-Latenz gegenüber Referenz": "p95_latenz",
    "Mittlere Latenz gegenüber Referenz": "mittlere_latenz",
}

zeilen = [
    json.loads(zeile)
    for zeile in open(ROOT / "export_kpis" / "dataset.jsonl", encoding="utf-8")
]
analyse_eval = [zeile for zeile in zeilen if zeile["split"] == "eval"]

saetze_eval = []
for zeile in analyse_eval:
    messung = {
        "target_method": zeile["method"],
        "env": zeile["env"],
        "variante": zeile["variant"],
        **zeile["metrics"],
    }
    antwort = bewerte(messung, referenz, notgrenzen)
    kriterien = {k["name"]: k for k in antwort["kriterien"]}
    verluste = sorted(
        (
            (k["gewicht"] * (100 - k["punkte"]), KURZNAMEN_EVAL[k["name"]])
            for k in antwort["kriterien"]
        ),
        reverse=True,
    )
    saetze_eval.append(
        {
            "variante": zeile["variant"],
            "methode": zeile["method"],
            "nummer": zeile["pos"],
            "umgebung": zeile["env"],
            "label_kpi": zeile["label"],
            "score_prozent": antwort["score_prozent"],
            "ziel_einstufung": antwort["einstufung"],
            "ziel_auffaellig": int(antwort["score_prozent"] < 75),
            "ziel_kriterium": verluste[0][1] if verluste[0][0] > 0 else "keines",
            "durchsatz_punkte": kriterien["Durchsatz gegenüber Referenz"]["punkte"],
            "p95_punkte": kriterien["p95-Latenz gegenüber Referenz"]["punkte"],
            "avg_punkte": kriterien["Mittlere Latenz gegenüber Referenz"]["punkte"],
            "durchsatz_abweichung": kriterien["Durchsatz gegenüber Referenz"][
                "abweichung_prozent"
            ],
            "p95_abweichung": kriterien["p95-Latenz gegenüber Referenz"][
                "abweichung_prozent"
            ],
            "avg_abweichung": kriterien["Mittlere Latenz gegenüber Referenz"][
                "abweichung_prozent"
            ],
            "snippet": zeile["snippet"],
        }
    )

daten_eval = pd.DataFrame(saetze_eval)

print(
    "Zeilen im Evaluationssatz:",
    len(daten_eval),
    "| davon mit Code:",
    int((daten_eval["snippet"].str.len() > 0).sum()),
)
print(
    "Zeilen je Variante:",
    dict(sorted(daten_eval["variante"].value_counts().to_dict().items())),
)
print()
print("Einstufung nach KPI-Label (0 = gesund, 1 = mutiert):")
print(pd.crosstab(daten_eval["label_kpi"], daten_eval["ziel_einstufung"]))
print()
print("Binäres Ziel (auffällig = Punktwert unter 75):")
print(
    daten_eval["ziel_auffaellig"]
    .value_counts()
    .rename({0: "unauffällig", 1: "auffällig"})
)
print("Prävalenz auffällig:", round(daten_eval["ziel_auffaellig"].mean(), 3))
print()
print("Auffällige je Projekt:")
print(daten_eval.groupby("variante")["ziel_auffaellig"].sum().to_string())

ziel_datei_eval = ROOT / "export_kpis" / "dataset_mit_labels_eval.csv"
daten_eval.to_csv(ziel_datei_eval, index=False)
print()
print("Gespeichert:", ziel_datei_eval)


Zeilen im Evaluationssatz: 230 | davon mit Code: 230
Zeilen je Variante: {'P12': 85, 'P12_neg': 15, 'P13': 100, 'P13_neg': 30}

Einstufung nach KPI-Label (0 = gesund, 1 = mutiert):
ziel_einstufung  Grenzfall  instabil  stabil
label_kpi                                   
0                        0         0     185
1                        5        25      15

Binäres Ziel (auffällig = Punktwert unter 75):
ziel_auffaellig
unauffällig    200
auffällig       30
Name: count, dtype: int64
Prävalenz auffällig: 0.13

Auffällige je Projekt:
variante
P12         0
P12_neg    15
P13         0
P13_neg    15

Gespeichert: /Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/export_kpis/dataset_mit_labels_eval.csv


13. Variante B: nur der Code angewandt auf das Trainingsdatenset

Jetzt kommt die eigentliche Frage der Arbeit. Bisher wurde aus Messwerten gelernt - das Modell sieht die KPI und sagt, ob ein Lauf gesund oder mutiert ist. Das nützt beim Pull Request nichts, weil es dort noch keine Messung gibt.

Variante B bekommt deshalb ausschließlich den Code zu sehen: das Snippet der geänderten Methode, sonst nichts. Vorhergesagt wird das Etikett aus der Matrix - also die Frage, ob diese Änderung auffällig werden wird.

Als Merkmale dient der Snippet-Text über Zeichenfolgen von drei bis fünf Zeichen. Quellcode variiert stark in Einrückung, Aufrufketten und Bezeichnern, weshalb Zeichenfolgen robuster sind als Wörter.

Gerechnet wird mit vier Gruppierungen, und das ist hier der entscheidende Punkt. Die Gruppierung legt fest, was zusammenbleiben muss: nach der Methodennummer die Zeilen einer Methode, nach dem Projekt Original und zugehörige Mutation desselben Projekts - sonst lernt das Modell den Projektstil statt das Fehlermuster. Nach der Variantenkennung wird diese Einheit bewusst zerrissen, um zu sehen, wie viel dann noch übrig bleibt. Für Methode und Projekt wird zehnmal zufällig aufgeteilt, für Projekt und Variante in fünf Faltungen; die Ausgabe umfasst damit vier Blöcke. Angegeben werden PR-AUC, Zufallswert, Präzision, Trefferquote und F1 je Konfiguration, dazu die tragenden Zeichenfolgen.

Zum Vergleich steht die KPI-Baseline aus Variante A mit 0,526 +- 0.178 auf zehn Aufteilungen und bis 0,638 aus der fünffachen Kreuzvalidierung bei einem Zufallswert von 0,165.

**Abgrenzung zu Variante A.** Das Modell ist ein anderes: Dort rechnet **LightGBM** auf den Messwerten, hier eine **logistische Regression** auf Zeichenfolgen des Code-Textes. Die eine Zahl ist deshalb nicht mit der anderen zu vergleichen, sondern jede nur mit ihrem eigenen Zufallswert – hier den Zeilenanteil der auffälligen Fälle.


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline

daten = pd.read_csv(
    ROOT / "export_kpis" / "dataset_mit_labels.csv", dtype={"nummer": str}
)
daten["projekt"] = daten["variante"].str.replace("_neg", "", regex=False)
ZIEL = "ziel_auffaellig"
y = daten[ZIEL].to_numpy()
X = daten["snippet"]


def baue_code_modell():
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2),
            ),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    )


def kennzahlen(gruppe, verfahren):
    if verfahren == "faltungen":
        teiler = list(
            GroupKFold(n_splits=5).split(X, y, groups=daten[gruppe].to_numpy())
        )
    else:
        teiler = [
            next(
                GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
                    X, y, groups=daten[gruppe].to_numpy()
                )
            )
            for seed in range(1, 11)
        ]
    zeilen = []
    for train_teil, test_teil in teiler:
        if y[test_teil].sum() == 0:
            continue
        modell = baue_code_modell()
        modell.fit(X.iloc[train_teil], y[train_teil])
        scores = modell.predict_proba(X.iloc[test_teil])[:, 1]
        precision, recall, schwellen = precision_recall_curve(y[test_teil], scores)
        geeignet = recall[:-1] >= 0.8
        schwelle = schwellen[geeignet].max() if geeignet.any() else 1.0
        vorhersage = (scores >= schwelle).astype(int)
        zeilen.append(
            {
                "PR-AUC": average_precision_score(y[test_teil], scores),
                "Zufall": y[test_teil].mean(),
                "Praezision": precision_score(
                    y[test_teil], vorhersage, zero_division=0
                ),
                "Trefferquote": recall_score(y[test_teil], vorhersage, zero_division=0),
                "F1": f1_score(y[test_teil], vorhersage, zero_division=0),
            }
        )
    return pd.DataFrame(zeilen)


print(
    "Ziel:",
    ZIEL,
    "| auffällig:",
    int(y.sum()),
    "von",
    len(y),
    "| Prävalenz:",
    round(y.mean(), 3),
)
print(
    "Gruppen: Methode",
    daten["nummer"].nunique(),
    "| Projekt",
    daten["projekt"].nunique(),
    "| Variante",
    daten["variante"].nunique(),
)
print()

for gruppe, verfahren, name in [
    ("nummer", "aufteilungen", "Gruppe Methode, 10 Aufteilungen"),
    ("projekt", "aufteilungen", "Gruppe Projekt projekt-rein, 10 Aufteilungen"),
    ("projekt", "faltungen", "Gruppe Projekt projekt-rein, 5 Faltungen"),
    ("variante", "faltungen", "Gruppe Variante, 5 Faltungen"),
]:
    tabelle = kennzahlen(gruppe, verfahren)
    print(f"{name}")
    print(
        f"   PR-AUC {round(tabelle['PR-AUC'].mean(), 3)} +/- {round(tabelle['PR-AUC'].std(), 3)}"
        f" | Zufallswert {round(tabelle['Zufall'].mean(), 3)}"
        f" | Präzision {round(tabelle['Praezision'].mean(), 3)}"
        f" | Trefferquote {round(tabelle['Trefferquote'].mean(), 3)}"
        f" | F1 {round(tabelle['F1'].mean(), 3)}"
    )
    print(f"   Einzelwerte PR-AUC: {[round(wert, 3) for wert in tabelle['PR-AUC']]}")
    print()

gesamt = baue_code_modell()
gesamt.fit(X, y)
merkmalnamen = gesamt.named_steps["tfidf"].get_feature_names_out()
koeffizienten = gesamt.named_steps["clf"].coef_[0]
sortiert = np.argsort(koeffizienten)
print("Stärkste Zeichenfolgen für 'auffällig':")
for i in sortiert[-15:][::-1]:
    print("   ", repr(merkmalnamen[i]), round(koeffizienten[i], 3))
print("Stärkste Zeichenfolgen für 'unauffällig':")
for i in sortiert[:10]:
    print("   ", repr(merkmalnamen[i]), round(koeffizienten[i], 3))

Ziel: ziel_auffaellig | auffällig: 68 von 485 | Prävalenz: 0.14
Gruppen: Methode 81 | Projekt 10 | Variante 18

Gruppe Methode, 10 Aufteilungen
   PR-AUC 0.28 +/- 0.112 | Zufallswert 0.123 | Präzision 0.224 | Trefferquote 0.951 | F1 0.356
   Einzelwerte PR-AUC: [0.29, 0.322, 0.366, 0.25, 0.347, 0.496, 0.167, 0.251, 0.216, 0.097]

Gruppe Projekt projekt-rein, 10 Aufteilungen
   PR-AUC 0.313 +/- 0.13 | Zufallswert 0.121 | Präzision 0.173 | Trefferquote 0.965 | F1 0.287
   Einzelwerte PR-AUC: [0.348, 0.133, 0.133, 0.352, 0.292, 0.393, 0.216, 0.543, 0.286, 0.431]

Gruppe Projekt projekt-rein, 5 Faltungen
   PR-AUC 0.361 +/- 0.148 | Zufallswert 0.135 | Präzision 0.218 | Trefferquote 0.978 | F1 0.352
   Einzelwerte PR-AUC: [0.251, 0.333, 0.414, 0.588, 0.217]

Gruppe Variante, 5 Faltungen
   PR-AUC 0.178 +/- 0.092 | Zufallswert 0.235 | Präzision 0.241 | Trefferquote 0.972 | F1 0.376
   Einzelwerte PR-AUC: [0.121, 0.285, 0.129]

Stärkste Zeichenfolgen für 'auffällig':
    'file' 0.494
    'ofi

14. Variante C: Code plus billige Messung (Stufe low)


Wenn schon eine Vorhersage ohne Messung möglich ist, dann interessiert die Frage, wie viel eine billige Messung zusätzlich bringt. Die Stufe low läuft schnell und kostet wenig. Variante C bekommt deshalb das Snippet, die Messwerte der Stufe low und die Zielumgebungsstufe - vorhergesagt wird das Etikett für die höhere Stufe.

Wichtig für die Sauberkeit: Als Merkmale dienen ausschließlich die Messwerte der Stufe low, nicht die der Zielstufe. Sonst würde die Antwort in der Eingabe stehen, weil das Etikett aus der Messung der Zielstufe berechnet wird.

Gerechnet werden drei Konfigurationen mit zwei Gruppierungen: nur die Low-Messung mit Zielumgebung, nur der Code und beides zusammen, jeweils über die Methode und über das Projekt.

Dazu kommt eine parameterfreie Regel als härteste Vergleichslinie: Ist der Lauf schon bei low auffällig, wird er es auch höher sein. Regeln ohne gelernte Parameter brauchen keine Aufteilung und können deshalb nicht durch eine günstige Gruppierung besser aussehen, als sie sind.

**Abgrenzung.** Variante C ist keine eigene Modellfamilie, sondern eine Zusammenführung: der **Code aus Variante B** und die **Messwerte der Stufe low**, in drei Kombinationen gegeneinander gestellt. Als Klassifikator dient hier durchweg eine logistische Regression, nicht das LightGBM aus Variante A. Gerechnet wird auf den Trainingsprojekten und auf den Evaluationsprojekten  P12 und P13.


In [19]:
from sklearn.metrics import confusion_matrix

zeilen_d = [
    json.loads(zeile)
    for zeile in open(ROOT / "export_kpis" / "dataset.jsonl", encoding="utf-8")
]
metriken = pd.DataFrame(
    [
        {
            "variante": zeile["variant"],
            "nummer": str(zeile["pos"]).zfill(3),
            "umgebung": zeile["env"],
            "rps": zeile["metrics"]["requests_per_sec"],
            "avg": zeile["metrics"]["avg_latency_ms"],
            "p95": zeile["metrics"]["p95_latency_ms"],
        }
        for zeile in zeilen_d
        if zeile["snippet"]
    ]
)
niedrig = metriken[metriken["umgebung"] == "low"][
    ["variante", "nummer", "rps", "avg", "p95"]
].rename(columns={"rps": "rps_low", "avg": "avg_low", "p95": "p95_low"})

daten_d = pd.read_csv(
    ROOT / "export_kpis" / "dataset_mit_labels.csv", dtype={"nummer": str}
)
daten_d["projekt"] = daten_d["variante"].str.replace("_neg", "", regex=False)
daten_d = (
    daten_d[daten_d["umgebung"].isin(["medium", "high", "extreme", "prod"])]
    .merge(niedrig, on=["variante", "nummer"], how="left")
    .reset_index(drop=True)
)

print(
    "Zeilen:",
    len(daten_d),
    "| auffällig:",
    int(daten_d[ZIEL].sum()),
    "| Prävalenz:",
    round(daten_d[ZIEL].mean(), 3),
)
print()

y_d = daten_d[ZIEL].to_numpy()
KPI_LOW = ["rps_low", "avg_low", "p95_low"]


def modell_d(mit_text, mit_kpi, mit_umgebung):
    teile = []
    if mit_text:
        teile.append(
            (
                "text",
                TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2),
                "snippet",
            )
        )
    if mit_kpi:
        teile.append(
            (
                "kpi",
                Pipeline(
                    steps=[
                        (
                            "log",
                            FunctionTransformer(
                                np.log1p, feature_names_out="one-to-one"
                            ),
                        ),
                        ("skala", StandardScaler()),
                    ]
                ),
                KPI_LOW,
            )
        )
    if mit_umgebung:
        teile.append(("umgebung", OneHotEncoder(handle_unknown="ignore"), ["umgebung"]))
    return Pipeline(
        steps=[
            ("merkmale", ColumnTransformer(transformers=teile)),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    )


def pruefe_d(mit_text, mit_kpi, mit_umgebung, gruppe, verfahren, beschriftung):
    spalten = (
        (["snippet"] if mit_text else [])
        + (KPI_LOW if mit_kpi else [])
        + (["umgebung"] if mit_umgebung else [])
    )
    X = daten_d[spalten]
    if verfahren == "faltungen":
        teiler = list(
            GroupKFold(n_splits=5).split(X, y_d, groups=daten_d[gruppe].to_numpy())
        )
    else:
        teiler = [
            next(
                GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
                    X, y_d, groups=daten_d[gruppe].to_numpy()
                )
            )
            for seed in range(1, 11)
        ]
    werte = []
    for train_teil, test_teil in teiler:
        if y_d[test_teil].sum() == 0:
            continue
        modell = modell_d(mit_text, mit_kpi, mit_umgebung)
        modell.fit(X.iloc[train_teil], y_d[train_teil])
        werte.append(
            average_precision_score(
                y_d[test_teil], modell.predict_proba(X.iloc[test_teil])[:, 1]
            )
        )
    print(
        f"{beschriftung:46s} {gruppe:8s} {verfahren:12s} PR-AUC {round(np.mean(werte), 3)} +/- {round(np.std(werte), 3)} (n={len(werte)})"
    )


print("Projekt-reine Gruppierung (5 Faltungen):")
pruefe_d(False, True, True, "projekt", "faltungen", "Nur Low-Messung + Zielumgebung")
pruefe_d(True, False, False, "projekt", "faltungen", "Nur Code")
pruefe_d(True, True, True, "projekt", "faltungen", "Code + Low-Messung + Zielumgebung")
print()
print("Gruppierung über die Methode (10 Aufteilungen):")
pruefe_d(False, True, True, "nummer", "aufteilungen", "Nur Low-Messung + Zielumgebung")
pruefe_d(True, False, False, "nummer", "aufteilungen", "Nur Code")
pruefe_d(
    True, True, True, "nummer", "aufteilungen", "Code + Low-Messung + Zielumgebung"
)
print()

low_etikett = (
    pd.read_csv(ROOT / "export_kpis" / "dataset_mit_labels.csv", dtype={"nummer": str})
    .query("umgebung == 'low'")[["variante", "nummer", ZIEL]]
    .rename(columns={ZIEL: "low_auffaellig"})
)
regel = daten_d.merge(low_etikett, on=["variante", "nummer"], how="left")

print("Parameterfreie Regel: Etikett der Low-Stufe auf die höhere Stufe übertragen")
print(
    pd.crosstab(
        regel["low_auffaellig"],
        regel[ZIEL],
        rownames=["low auffällig"],
        colnames=["Ziel auffällig"],
    )
)
tn, fp, fn, tp = confusion_matrix(regel[ZIEL], regel["low_auffaellig"]).ravel()
print()
print(
    "  Treffer:",
    tp,
    "| Fehlalarm:",
    fp,
    "| übersehen:",
    fn,
    "| korrekt freigegeben:",
    tn,
)
print(
    "  Präzision:",
    round(tp / (tp + fp), 3),
    "| Trefferquote:",
    round(tp / (tp + fn), 3),
    "| F1:",
    round(2 * tp / (2 * tp + fp + fn), 3),
)

print()
print("Parameterfreie Regel je Zielstufe:")
for stufe in ("medium", "high", "extreme", "prod"):
    teil = regel[regel["umgebung"] == stufe]
    tn_s, fp_s, fn_s, tp_s = confusion_matrix(
        teil[ZIEL], teil["low_auffaellig"], labels=[0, 1]
    ).ravel()
    print(
        f"  {stufe:8s} n {len(teil):3d} | auffällig {int(teil[ZIEL].sum()):2d}"
        f" | Treffer {tp_s:2d} | Fehlalarm {fp_s:2d} | übersehen {fn_s:2d}"
        f" | korrekt freigegeben {tn_s:3d}"
    )
    spalten_d = ["snippet", "rps_low", "avg_low", "p95_low", "umgebung"]
X_d = daten_d[spalten_d]
prognose_d = np.full(len(daten_d), np.nan)
for train_teil, test_teil in GroupKFold(n_splits=5).split(
    X_d, y_d, groups=daten_d["projekt"].to_numpy()
):
    modell = modell_d(True, True, True)
    modell.fit(X_d.iloc[train_teil], y_d[train_teil])
    prognose_d[test_teil] = modell.predict_proba(X_d.iloc[test_teil])[:, 1]

print()
print("Code + Low-Messung + Zielumgebung, Out-of-Fold je Zielstufe:")
for stufe in ("medium", "high", "extreme", "prod"):
    maske = (daten_d["umgebung"] == stufe).to_numpy()
    y_stufe = y_d[maske]
    p_stufe = prognose_d[maske]
    print(
        f"  {stufe:8s} n {int(maske.sum()):3d} | auffällig {int(y_stufe.sum()):2d}"
        f" | mittleres p auffällig {round(p_stufe[y_stufe == 1].mean(), 3)}"
        f" | mittleres p unauffällig {round(p_stufe[y_stufe == 0].mean(), 3)}"
    )

Zeilen: 388 | auffällig: 55 | Prävalenz: 0.142

Projekt-reine Gruppierung (5 Faltungen):
Nur Low-Messung + Zielumgebung                 projekt  faltungen    PR-AUC 0.521 +/- 0.282 (n=5)
Nur Code                                       projekt  faltungen    PR-AUC 0.375 +/- 0.177 (n=5)
Code + Low-Messung + Zielumgebung              projekt  faltungen    PR-AUC 0.562 +/- 0.302 (n=5)

Gruppierung über die Methode (10 Aufteilungen):
Nur Low-Messung + Zielumgebung                 nummer   aufteilungen PR-AUC 0.485 +/- 0.172 (n=10)
Nur Code                                       nummer   aufteilungen PR-AUC 0.282 +/- 0.106 (n=10)
Code + Low-Messung + Zielumgebung              nummer   aufteilungen PR-AUC 0.653 +/- 0.245 (n=10)

Parameterfreie Regel: Etikett der Low-Stufe auf die höhere Stufe übertragen
Ziel auffällig    0   1
low auffällig          
0               328   8
1                 5  47

  Treffer: 47 | Fehlalarm: 5 | übersehen: 8 | korrekt freigegeben: 328
  Präzision: 0.904 | Treff



15. Evaluationsdatenset: Variante A und Variante B auf P12 und P13

 Bei Modelle Variant A LightGbm und Variante B die logistische Regression auf Zeichenfolgen werden auf P12 und P13 angewandt, also auf zwei Projekte, die bei keiner Entscheidung mitgewirkt haben.

Variante A wird zuerst thresholdfrei bewertet: Die PR-AUC sagt, wie gut das Modell die Zeilen ordnet, unabhängig von jedem Schnitt. Danach wird es mit der Grenze aus Abschnitt 11 angewandt. Diese Grenze gehört zum dort fertiggestellten Modell: Sie wurde aus den Trainingszeilen für eine Trefferquote von 0,8 bestimmt und liegt bei 0,027. Auf dem Evaluatonsdatenset wird an ihr nichts geändert - würde man sie hier nachziehen, wäre das Ergebnis in die Einstellung eingerechnet und die Prüfung wertlos. Dass die Trefferquote dabei von 0,800 auf 0,511 fällt, ist deshalb ein Befund über die Übertragbarkeit und kein Einstellfehler.

Variante B wird thresholdfrei bewertet. Ihr Etikett ist dasselbe wie bei Variante C, nämlich die Auffälligkeit aus der Bewertungsmatrix; ihr Merkmal ist ausschließlich der Code. Vergleichsgröße ist in beiden Fällen die Prävalenz, also der Anteil der auffälligen Zeilen - der Wert, den ein Verfahren ohne jede Modellleistung erreicht.

Die beiden Varianten beantworten verschiedene Fragen: Variante A bewertet Messwerte und sagt, ob die Punktzahl aus der Matrix vertrauenswürdig ist; Variante B bewertet nur den Code und sagt, ob sich eine Änderung schon vor der Messung ankündigt. Deshalb stehen sie nebeneinander und werden nicht zu einer Kennzahl verrechnet.

Ein Hinweis zur Aussagekraft: Die Stichprobe ist klein, neun Fälle in fünf Stufen. Einzelne Prozentpunkte sind deshalb nicht belastbar; belastbar ist die Richtung, also ob die PR-AUC deutlich über der Prävalenz liegt oder nicht.


In [20]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupKFold

X_train_final = ohne_grenzfaelle[FEATURES_FINAL]
y_train_final = ohne_grenzfaelle[TARGET].to_numpy()
gruppen_train = ohne_grenzfaelle["pair_id"].to_numpy()

ziel_trefferquote = 0.8

scores_kreuz = np.zeros(len(ohne_grenzfaelle))
for train_teil, test_teil in GroupKFold(n_splits=5).split(
    X_train_final, y_train_final, groups=gruppen_train
):
    teil_modell = modell_bauen(FEATURES_FINAL, False)
    teil_modell.fit(X_train_final.iloc[train_teil], y_train_final[train_teil])
    scores_kreuz[test_teil] = teil_modell.predict_proba(X_train_final.iloc[test_teil])[
        :, 1
    ]

precision, recall, schwellen = precision_recall_curve(y_train_final, scores_kreuz)
grenze = schwellen[recall[:-1] >= ziel_trefferquote].max()

modell_kpi = modell_bauen(FEATURES_FINAL, False)
modell_kpi.fit(X_train_final, y_train_final)

eval_kpi = df[df["project_id"].isin(EVAL_PROJECTS)].copy()
y_eval_kpi = eval_kpi[TARGET].to_numpy()
prognose_kpi = modell_kpi.predict_proba(eval_kpi[FEATURES_FINAL])[:, 1]

print("Variante A (LightGBM) auf dem Evaluationsdatenset, P12 und P13")
print(
    "  Zeilen:",
    len(eval_kpi),
    "| mutiert:",
    int(y_eval_kpi.sum()),
    "| Prävalenz:",
    round(y_eval_kpi.mean(), 3),
)
print("  PR-AUC:", round(average_precision_score(y_eval_kpi, prognose_kpi), 3))
print()
print("  Entscheidungsgrenze aus Abschnitt 11:", round(grenze, 3))
vorhersage_kpi = (prognose_kpi >= grenze).astype(int)
tn, fp, fn, tp = confusion_matrix(y_eval_kpi, vorhersage_kpi).ravel()
print(
    "  Precision:",
    round(precision_score(y_eval_kpi, vorhersage_kpi, zero_division=0), 3),
    "| Trefferquote:",
    round(recall_score(y_eval_kpi, vorhersage_kpi, zero_division=0), 3),
    "| F1:",
    round(f1_score(y_eval_kpi, vorhersage_kpi, zero_division=0), 3),
)
print("  Verwirrungsmatrix  TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)
print()

eval_daten = pd.read_csv(
    ROOT / "export_kpis" / "dataset_mit_labels_eval.csv", dtype={"nummer": str}
)
eval_code = eval_daten[eval_daten["snippet"].str.len() > 0].reset_index(drop=True)
y_eval_code = eval_code["ziel_auffaellig"].to_numpy()

modell_code = baue_code_modell()
modell_code.fit(daten["snippet"], daten["ziel_auffaellig"])
prognose_code = modell_code.predict_proba(eval_code["snippet"])[:, 1]

print("Variante B: Code-Modell auf dem Evaluationsdatenset")
print(
    "  Zeilen:",
    len(eval_code),
    "| auffällig:",
    int(y_eval_code.sum()),
    "| Prävalenz:",
    round(y_eval_code.mean(), 3),
)
print("  PR-AUC:", round(average_precision_score(y_eval_code, prognose_code), 3))
print(
    "  Mittleres p bei auffällig:",
    round(prognose_code[y_eval_code == 1].mean(), 3) if y_eval_code.sum() else "keine",
)
print(
    "  Mittleres p bei unauffällig:", round(prognose_code[y_eval_code == 0].mean(), 3)
)

Variante A (LightGBM) auf dem Evaluationsdatenset, P12 und P13
  Zeilen: 230 | mutiert: 45 | Prävalenz: 0.196
  PR-AUC: 0.469

  Entscheidungsgrenze aus Abschnitt 11: 0.027
  Precision: 0.284 | Trefferquote: 0.511 | F1: 0.365
  Verwirrungsmatrix  TN: 127 FP: 58 FN: 22 TP: 23

Variante B: Code-Modell auf dem Evaluationsdatenset
  Zeilen: 230 | auffällig: 30 | Prävalenz: 0.13
  PR-AUC: 0.189
  Mittleres p bei auffällig: 0.117
  Mittleres p bei unauffällig: 0.107


16. Variante C auf Projekt P12 und P13

Dasselbe verfahren  wie in Abschnitt 14, aber ohne Kreuzvalidierung: Trainiert wird einmal auf den Trainingsprojekten P01 bis P11, vorhergesagt werden die Evaluationszeilen aus Abschnitt 12 B. Mit zwei Projekten lässt sich keine projekt-reine Aufteilung bilden, es bleibt bei einem einmaligen Transfer.

Die Merkmale der Prüfzeilen stammen weiterhin ausschließlich aus Snippet und der Messung der Stufe low. Die Bewertung der Zielstufe, aus der das Etikett berechnet wird, bleibt außen vor.

Der Vergleich mit Variante B ist der Ertrag dieser Zelle: Beide bekommen denselben Evaluationsdatenset und dasselbe Etikett, B aber nur den Code. Trägt die Kombination mehr als der Code allein, trägt die billige Messung etwas bei. Die Vergleichszahl für B steht in der Ausgabe von Abschnitt 15.

In [21]:
zeilen_eval_c = [
    json.loads(zeile)
    for zeile in open(ROOT / "export_kpis" / "dataset.jsonl", encoding="utf-8")
]
eval_metriken = pd.DataFrame(
    [
        {
            "variante": zeile["variant"],
            "nummer": str(zeile["pos"]).zfill(3),
            "umgebung": zeile["env"],
            "rps": zeile["metrics"]["requests_per_sec"],
            "avg": zeile["metrics"]["avg_latency_ms"],
            "p95": zeile["metrics"]["p95_latency_ms"],
        }
        for zeile in zeilen_eval_c
        if zeile["snippet"]
    ]
)
eval_niedrig = eval_metriken[eval_metriken["umgebung"] == "low"][
    ["variante", "nummer", "rps", "avg", "p95"]
].rename(columns={"rps": "rps_low", "avg": "avg_low", "p95": "p95_low"})

daten_eval_c = pd.read_csv(
    ROOT / "export_kpis" / "dataset_mit_labels_eval.csv", dtype={"nummer": str}
)
daten_eval_c["projekt"] = daten_eval_c["variante"].str.replace("_neg", "", regex=False)
daten_eval_c = (
    daten_eval_c[daten_eval_c["umgebung"].isin(["medium", "high", "extreme", "prod"])]
    .merge(eval_niedrig, on=["variante", "nummer"], how="left")
    .reset_index(drop=True)
)

print(
    "Zeilen:",
    len(daten_eval_c),
    "| auffällig:",
    int(daten_eval_c[ZIEL].sum()),
    "| Prävalenz:",
    round(daten_eval_c[ZIEL].mean(), 3),
)
print()

y_eval_c = daten_eval_c[ZIEL].to_numpy()


def pruefe_c_Evaluationsdatenset(mit_text, mit_kpi, mit_umgebung, beschriftung):
    spalten = (
        (["snippet"] if mit_text else [])
        + (KPI_LOW if mit_kpi else [])
        + (["umgebung"] if mit_umgebung else [])
    )
    modell = modell_d(mit_text, mit_kpi, mit_umgebung)
    modell.fit(daten_d[spalten], y_d)
    scores = modell.predict_proba(daten_eval_c[spalten])[:, 1]
    print(
        f"{beschriftung:46s} PR-AUC {round(average_precision_score(y_eval_c, scores), 3)}"
        f" | Zufallswert {round(y_eval_c.mean(), 3)}"
    )


print("Trainiert auf P01 bis P11, vorhergesagt für P12 und P13:")
pruefe_c_Evaluationsdatenset(False, True, True, "Nur Low-Messung + Zielumgebung")
pruefe_c_Evaluationsdatenset(True, False, False, "Nur Code")
pruefe_c_Evaluationsdatenset(True, True, True, "Code + Low-Messung + Zielumgebung")


low_etikett_eval = (
    pd.read_csv(
        ROOT / "export_kpis" / "dataset_mit_labels_eval.csv", dtype={"nummer": str}
    )
    .query("umgebung == 'low'")[["variante", "nummer", ZIEL]]
    .rename(columns={ZIEL: "low_auffaellig"})
)
regel_eval = daten_eval_c.merge(low_etikett_eval, on=["variante", "nummer"], how="left")

print()
print("Parameterfreie Regel: Etikett der Low-Stufe auf die höhere Stufe übertragen")
print(
    pd.crosstab(
        regel_eval["low_auffaellig"],
        regel_eval[ZIEL],
        rownames=["low auffällig"],
        colnames=["Ziel auffällig"],
    )
)
tn_r, fp_r, fn_r, tp_r = confusion_matrix(
    regel_eval[ZIEL], regel_eval["low_auffaellig"], labels=[0, 1]
).ravel()
print()
print(
    "  Treffer:",
    tp_r,
    "| Fehlalarm:",
    fp_r,
    "| übersehen:",
    fn_r,
    "| korrekt freigegeben:",
    tn_r,
)
print(
    "  Präzision:",
    round(tp_r / (tp_r + fp_r), 3),
    "| Trefferquote:",
    round(tp_r / (tp_r + fn_r), 3),
    "| F1:",
    round(2 * tp_r / (2 * tp_r + fp_r + fn_r), 3),
)

Zeilen: 184 | auffällig: 26 | Prävalenz: 0.141

Trainiert auf P01 bis P11, vorhergesagt für P12 und P13:
Nur Low-Messung + Zielumgebung                 PR-AUC 0.476 | Zufallswert 0.141
Nur Code                                       PR-AUC 0.218 | Zufallswert 0.141
Code + Low-Messung + Zielumgebung              PR-AUC 0.423 | Zufallswert 0.141

Parameterfreie Regel: Etikett der Low-Stufe auf die höhere Stufe übertragen
Ziel auffällig    0   1
low auffällig          
0               158  10
1                 0  16

  Treffer: 16 | Fehlalarm: 0 | übersehen: 10 | korrekt freigegeben: 158
  Präzision: 1.0 | Trefferquote: 0.615 | F1: 0.762


17. Beiträge des Endmodells (Grundlage der Modell-Erklärung)

Diese Zelle erklärt nicht die Bewertungsmatrix, sondern das Modell. Sie hält fest, in welcher Richtung und ab welcher Größenordnung jedes der drei Merkmale die Vorhersage verschiebt, und schreibt die Bereiche samt mittlerem Beitrag als `prototype/beitraege.csv` weg. Die Datei ist der Eingang für die Modelleinschätzung im Dashboard. 

In [22]:
import shap 
daten_beitraege = ohne_grenzfaelle[FEATURES_FINAL].copy()
modell_beitraege = modell_bauen(FEATURES_FINAL,False)
modell_beitraege.fit(daten_beitraege, ohne_grenzfaelle[TARGET])

vorbereitete_daten = modell_beitraege.named_steps["prep"].transform(daten_beitraege)
feature_namen = list(modell_beitraege.named_steps["prep"].get_feature_names_out())
erklaerer = shap.TreeExplainer(modell_beitraege.named_steps["clf"])
werte = erklaerer.shap_values(vorbereitete_daten)
if isinstance(werte, list): 
    werte = werte[1]
elif getattr(werte,"ndim",2) == 3:
    werte = werte[:,:,1]
    
if werte.shape != (len(daten_beitraege), len(feature_namen)):
    raise systemexit(f"Unerwartete Form der Eintraege: {werte.shape}")

beitraege = pd.DataFrame(werte, columns=feature_namen, index=daten_beitraege.index)

zeilen = []
for merkmal in FEATURES_FINAL:
    bereiche = pd.qcut(daten_beitraege[merkmal],5,duplicates="drop")
    for _, gruppe in daten_beitraege.groupby(bereiche, observed=True):
        zeilen.append(
            {
                "merkmal": merkmal,
                "wert_von": round(float(gruppe[merkmal].min()),2),
                "wert_bis": round(float(gruppe[merkmal].max()),2),
                "beitrag_mittel": round(float(beitraege.loc[gruppe.index, merkmal].mean()),4),
                "zeilen": len(gruppe),                  
            }
        )

beitragstabelle = pd.DataFrame(zeilen)

print("Endmodel: LightGBM angewendet auf", FEATURES_FINAL)
print("Trainingszeilen:", len(daten_beitraege), "Bereiche gesamt:", len(beitragstabelle))

print()
print(beitragstabelle.to_string(index=False))

ziel = ROOT/ "prototype" / "beitraege.csv"
beitragstabelle.to_csv(ziel, index=False)

print()
print("Gespeichert in :", ziel)

Endmodel: LightGBM angewendet auf ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec']
Trainingszeilen: 485 Bereiche gesamt: 15

         merkmal  wert_von  wert_bis  beitrag_mittel  zeilen
  avg_latency_ms      5.30     58.70         -0.3382      97
  avg_latency_ms     58.93    318.62         -0.0166      97
  avg_latency_ms    320.36   2006.93          0.0516      97
  avg_latency_ms   2008.74   7309.73          0.0357      97
  avg_latency_ms   7481.16  44660.62          0.3426      97
  p95_latency_ms      7.00    240.00         -0.1525      97
  p95_latency_ms    250.00    500.00         -0.5926      99
  p95_latency_ms    510.00   3300.00          0.0042      95
  p95_latency_ms   3400.00  16000.00          1.0490      98
  p95_latency_ms  17000.00  79000.00         -0.1979      96
requests_per_sec      0.34      3.35          2.1564      98
requests_per_sec      3.36      6.96         -1.4556      97
requests_per_sec      7.02     40.62          1.0328      96
requests_per_

/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/.venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [23]:
daten_beitraege = ohne_grenzfaelle[FEATURES_FINAL].copy()
modell_beitraege = modell_bauen(FEATURES_FINAL, False)
modell_beitraege.fit(daten_beitraege, ohne_grenzfaelle[TARGET])

vorbereitete_daten = modell_beitraege.named_steps["prep"].transform(daten_beitraege)
feature_namen = list(modell_beitraege.named_steps["prep"].get_feature_names_out())
umgebungen = ohne_grenzfaelle["env"].to_numpy()

zeilen = []
erwartungswerte = {}
for stufe in ["low", "medium", "high", "extreme", "prod"]:
    maske = umgebungen == stufe
    stufe_daten = daten_beitraege.loc[maske]
    erklaerer = shap.TreeExplainer(
        modell_beitraege.named_steps["clf"],
        data=vorbereitete_daten[maske],
        feature_perturbation="interventional",
        feature_names=feature_namen,
    )
    werte = erklaerer.shap_values(vorbereitete_daten[maske])
    if isinstance(werte, list):
        werte = werte[1]
    elif getattr(werte, "ndim", 2) == 3:
        werte = werte[:, :, 1]
    if werte.shape != (len(stufe_daten), len(feature_namen)):
        raise SystemExit(f"Unerwartete Form der Beitraege in {stufe}: {werte.shape}")

    erwartungswerte[stufe] = float(np.ravel(erklaerer.expected_value)[-1])
    stufe_tabelle = stufe_daten.join(
        pd.DataFrame(
            werte,
            columns=[f"beitrag_{name}" for name in feature_namen],
            index=stufe_daten.index,
        )
    )

    for merkmal in FEATURES_FINAL:
        bereiche = pd.qcut(stufe_tabelle[merkmal], 5, duplicates="drop")
        for _, gruppe in stufe_tabelle.groupby(bereiche, observed=True):
            zeilen.append(
                {
                    "umgebung": stufe,
                    "merkmal": merkmal,
                    "wert_von": round(float(gruppe[merkmal].min()), 2),
                    "wert_bis": round(float(gruppe[merkmal].max()), 2),
                    "beitrag_mittel": round(float(gruppe[f"beitrag_{merkmal}"].mean()), 4),
                    "zeilen": len(gruppe),
                }
            )


beitragstabelle = pd.DataFrame(zeilen)

print("Endmodell: LightGBM auf", FEATURES_FINAL, "| Ziel:", TARGET)
print("Trainingszeilen gesamt:", len(daten_beitraege))
print()
print("Erwartungswert je Umgebung (Rohmarge des Hintergrunds):")
for stufe, wert in erwartungswerte.items():
    print(f"   {stufe:8s} {round(wert, 3)}")

for stufe in ["low", "medium", "high", "extreme", "prod"]:
    print()
    print(stufe)
    print(
        beitragstabelle[beitragstabelle["umgebung"] == stufe]
        .drop(columns="umgebung")
        .to_string(index=False)
    )

ziel = ROOT / "prototype" / "beitraege.csv"
beitragstabelle.to_csv(ziel, index=False)
print()
print("Gespeichert in:", ziel)

Endmodell: LightGBM auf ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec'] | Ziel: is_neg
Trainingszeilen gesamt: 485

Erwartungswert je Umgebung (Rohmarge des Hintergrunds):
   low      -2.965
   medium   -2.813
   high     -3.082
   extreme  -2.975
   prod     -2.864

low
         merkmal  wert_von  wert_bis  beitrag_mittel  zeilen
  avg_latency_ms      5.90     24.36         -0.7199      20
  avg_latency_ms     24.46     42.40         -0.4565      19
  avg_latency_ms     43.69    198.65          0.6902      19
  avg_latency_ms    420.47   2006.93          0.2904      19
  avg_latency_ms   2008.74  11996.58          0.2220      20
  p95_latency_ms      7.00     90.00         -0.0819      20
  p95_latency_ms     92.00    190.00         -0.6998      19
  p95_latency_ms    200.00    410.00         -1.1187      19
  p95_latency_ms    820.00   3400.00          0.9613      19
  p95_latency_ms   3600.00  19000.00          0.8963      20
requests_per_sec      0.40      3.65          0.

**Analyse Zelle 17: Shap Vergleich unter absoluten Werten**

Die Beiträge sind über die Wertebereiche hinweg **nicht geordnet**. Am deutlichsten beim Durchsatz: Der niedrigste Bereich (0,34 bis 3,35 Anfragen pro Sekunde) trägt mit **+2,16** den stärksten positiven Beitrag der Tabelle – das ist richtig, denn kaum Durchsatz bedeutet schweren Schaden. Der Bereich direkt darüber (3,36 bis 6,96) trägt dagegen **−1,46** und spricht damit für einen gesunden Lauf, obwohl der Durchsatz dort ebenfalls sehr niedrig ist. Auch der höchste Bereich (85,98 bis 234,94) liegt mit **−1,66** klar im Negativen. Positiv heißt in dieser Tabelle: der Wert spricht für „mutiert"; negativ heißt: er spricht für „gesund".

Das ist kein Rechenfehler. Die Auswertung der Bereiche zeigt den Grund: Sie enthalten **verschiedene Populationen**. Der Anteil mutierter Zeilen schwankt zwischen 0,021 und 0,429, und der Anteil der Zeilen aus `extreme` und `prod` liegt in den beiden Randbereichen bei 0,480 und 0,598, in den drei mittleren dagegen nur bei 0,278 bis 0,330. Ein niedriger Durchsatz bedeutet also je nach Herkunft etwas anderes: einen langsamen Endpunkt in ruhiger Umgebung oder einen eingebrochenen unter extremer Last. Die Beiträge folgen dabei den Mutationsanteilen der Bereiche, nicht dem Zahlenwert.

**Folge.** Ein absoluter Wertebereich ist über Laststufen hinweg nicht interpretierbar; die Tabelle darf nicht als „so wirkt dieser Wert" gelesen werden. Abschnitt 18 rechnet dieselbe Auswertung deshalb auf den **relativen** Abweichungen zur Referenz derselben Methode und Umgebungsstufe – dort bedeutet derselbe Wert in jeder Stufe dasselbe.